## **Requirements**

### Mode 1 — Local (VSCode only)

#### Java
PySpark requires Java to run. **Java 17 is required** — newer versions
(18+) are not fully compatible with the current Spark version.

1. Download and install **Eclipse Temurin JDK 17** from:  
   https://adoptium.net/temurin/releases/?version=17
2. After installing, set `JAVA_HOME` in Windows System Environment Variables
   to the JDK 17 installation path, e.g.:  
   `C:\Users\YourUser\AppData\Local\Programs\Eclipse Adoptium\jdk-17.x.x-hotspot`
3. Confirm the correct version is active:
```bash
   java -version
   # Expected: openjdk version "17.x.x"
```

#### Python
- Python 3.8 or higher

#### Libraries
```bash
pip install pyspark
```

---

### Mode 2 — Google Cloud Dataproc (recommended for performance)

Processes data in parallel across multiple machines — queries and ML 
models run significantly faster than local mode.

> ⚠️ **There is no pause option in Dataproc.** The recommended workflow 
> is to **create the cluster when you start working and delete it when 
> you finish**. The data stays safely in the Google Cloud Storage bucket 
> and is never lost. You are only charged while the cluster exists.

#### One-time setup
1. Create a **Google Cloud account** and activate billing
2. Upload the 12 Parquet files to a **Google Cloud Storage** bucket
3. Install **Google Cloud SDK**: https://cloud.google.com/sdk/docs/install
4. Initialise the SDK and set the default region:
```bash
   gcloud init
   # Select your project and choose europe-west1 as default region
```

---

#### Every time you want to work

**Step 1** — Create the cluster (~3 minutes):
```bash
gcloud dataproc clusters create cluster-taxi --region=europe-west1 --zone=europe-west1-b --master-machine-type=n2-standard-4 --worker-machine-type=n2-standard-4 --num-workers=3 --image-version=2.1
```

**Step 2** — Open a terminal and create the SSH tunnel (keep it open):
```bash
gcloud compute ssh cluster-taxi-m --zone=europe-west1-b -- -L 8888:localhost:8888
```

**Step 3** — In the SSH window that opens, start Jupyter:
```bash
jupyter notebook --no-browser --port=8888
```

**Step 4** — In the SSH window, after running the Jupyter command, 
a URL will appear automatically, e.g.:
http://localhost:8888/?token=YOUR_TOKEN_HERE

Copy this full URL — the token changes every time Jupyter starts.

**Step 5** — In VSCode, click the kernel selector (top right) →
**Select Another Kernel** → **Existing Jupyter Server** → paste the URL

**Step 6** — Work normally in VSCode. Your data is read directly 
from the bucket:
```python
files = [f"gs://YOUR_BUCKET_NAME/yellow_tripdata_2020-{str(i).zfill(2)}.parquet" 
         for i in range(1, 13)]
```

**Step 7** — When finished, **always delete the cluster** to avoid charges:
```bash
gcloud dataproc clusters delete cluster-taxi --region=europe-west1
```

In [57]:
# ── Standard Library ────────────────────────────────────────────
import time
import glob
from collections import Counter
from functools import reduce

# ── PySpark Core ─────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType, TimestampType
)
from pyspark.sql.functions import percentile_approx

# ── PySpark ML ───────────────────────────────────────────────────
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [48]:
spark = SparkSession.builder \
    .appName("AnalisePatricia") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(spark.sparkContext._jvm.System.getProperty("java.version"))

11.0.30


In [49]:
files = [f"gs://taxi-data-2020-patricia/yellow_tripdata_2020-{str(i).zfill(2)}.parquet" for i in range(1, 13)]
print(files)

['gs://taxi-data-2020-patricia/yellow_tripdata_2020-01.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-02.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-03.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-04.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-05.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-06.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-07.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-08.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-09.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-10.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-11.parquet', 'gs://taxi-data-2020-patricia/yellow_tripdata_2020-12.parquet']


In [50]:
df = spark.read.parquet(*files) \
    .withColumn("airport_fee", F.col("airport_fee").cast("double"))

#### **Schema Inconsistency — `airport_fee`**

The `airport_fee` column presented type inconsistencies across the 12 Parquet files (`INT32`, `DOUBLE`, and `VOID`), causing schema conversion errors at read time. Since this field was not applicable in 2020 — it was only introduced by the NYC TLC in 2022 — it was dropped immediately upon loading to avoid conflicts.

In [51]:
df = df.drop("airport_fee")

df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|1       |2020-01-01 00:28:15 |2020-01-01 00:33:03  |1.0            |1.2          |1.0       |N                 |238         |239         |1           |6.0        |3.0  |0.5    |1.47      |0.0         |0.3                  

In [52]:
for f in files:
    temp_df = spark.read.parquet(f).withColumn("airport_fee", F.col("airport_fee").cast("double"))
    print(f"{f.split('/')[-1]} → {temp_df.count()} linhas, {len(temp_df.columns)} colunas")

yellow_tripdata_2020-01.parquet → 6405008 linhas, 19 colunas
yellow_tripdata_2020-02.parquet → 6299367 linhas, 19 colunas
yellow_tripdata_2020-03.parquet → 3007687 linhas, 19 colunas
yellow_tripdata_2020-04.parquet → 238073 linhas, 19 colunas
yellow_tripdata_2020-05.parquet → 348415 linhas, 19 colunas
yellow_tripdata_2020-06.parquet → 549797 linhas, 19 colunas
yellow_tripdata_2020-07.parquet → 800412 linhas, 19 colunas
yellow_tripdata_2020-08.parquet → 1007286 linhas, 19 colunas
yellow_tripdata_2020-09.parquet → 1341017 linhas, 19 colunas
yellow_tripdata_2020-10.parquet → 1681132 linhas, 19 colunas
yellow_tripdata_2020-11.parquet → 1509000 linhas, 19 colunas
yellow_tripdata_2020-12.parquet → 1461898 linhas, 19 colunas


# **1. Dataset Overview**

### **Dimensions**

In [53]:
# ── 1. DIMENSÕES GERAIS ──────────────────────────────────────────
total_rows = df.count()
total_cols = len(df.columns)
print(f"Total de linhas: {total_rows:,}")
print(f"Total de colunas: {total_cols}")

Total de linhas: 24,649,092
Total de colunas: 18


### **Data Types**

In [54]:
# ── 2. SCHEMA POR TIPOS ─────────────────────────────────────────
type_counts = Counter([str(f.dataType) for f in df.schema.fields])
for t, count in type_counts.items():
    print(f"{t}: {count} colunas")

LongType(): 4 colunas
TimestampType(): 2 colunas
DoubleType(): 11 colunas
StringType(): 1 colunas


The dataset comprises **18 attributes** distributed across 4 data types:

- **LongType** (4 columns): Integer fields representing IDs and categorical codes — `VendorID`, `PULocationID`, `DOLocationID`, and `payment_type`.

- **TimestampNTZType** (2 columns): Datetime fields without timezone — `tpep_pickup_datetime` and `tpep_dropoff_datetime`.

- **DoubleType** (11 columns): Continuous numerical fields representing distances, fare components, and trip metadata.

- **StringType** (1 column): The `store_and_fwd_flag` field, indicating whether the trip record was temporarily stored before transmission (Y/N).

### **Dataset Attributes**

| Field Name | Type | Description |
|---|---|---|
| `VendorID` | Long | Identifies the technology provider that recorded the trip. **1** = Creative Mobile Technologies, **2** = VeriFone Inc. |
| `tpep_pickup_datetime` | Timestamp | Date and time when the taximeter was engaged (trip start). |
| `tpep_dropoff_datetime` | Timestamp | Date and time when the taximeter was disengaged (trip end). |
| `passenger_count` | Double | Number of passengers in the vehicle, manually entered by the driver. |
| `trip_distance` | Double | Total trip distance in miles as reported by the taximeter. |
| `RatecodeID` | Double | Rate code applied at the end of the trip. **1** = Standard, **2** = JFK, **3** = Newark, **4** = Nassau/Westchester, **5** = Negotiated, **6** = Group ride. |
| `store_and_fwd_flag` | String | Indicates whether the trip record was temporarily stored in the vehicle due to lack of server connection before being transmitted. **Y** = stored, **N** = not stored. |
| `PULocationID` | Long | TLC Taxi Zone ID where the passenger was picked up. |
| `DOLocationID` | Long | TLC Taxi Zone ID where the passenger was dropped off. |
| `payment_type` | Long | Payment method used. **1** = Credit card, **2** = Cash, **3** = No charge, **4** = Dispute, **5** = Unknown, **6** = Voided. |
| `fare_amount` | Double | Base fare calculated by the taximeter based on time and distance. |
| `extra` | Double | Miscellaneous surcharges, including **$0.50** rush hour and **$1.00** overnight supplements. |
| `mta_tax` | Double | Fixed **$0.50** tax automatically applied to all standard-rate trips, funding the Metropolitan Transportation Authority (NYC public transit). |
| `improvement_surcharge` | Double | Fixed **$0.30** surcharge levied at the start of every trip since 2015, funding wheelchair-accessible taxi improvements. |
| `tip_amount` | Double | Tip amount, automatically captured for credit card payments only. Cash tips are not recorded. |
| `tolls_amount` | Double | Total amount of road tolls paid during the trip (bridges, tunnels, etc.). |
| `total_amount` | Double | Total amount charged to the passenger, including all fees and surcharges. Does not include cash tips. |
| `congestion_surcharge` | Double | Surcharge of **$2.50** applied to trips entering or leaving Manhattan below 96th Street, introduced in 2019 to fund public transport improvements. Not applicable to all trips, hence the missing values. |
| `airport_fee` | Double | Fixed **$1.25** surcharge for pickups at LaGuardia (LGA) and John F. Kennedy (JFK) airports. **Not applicable in 2020** — introduced by NYC TLC in 2022. |

# **2. Data Quality Check**

### **Temporal Range Validation**

In [55]:
# ── 3. INTERVALO TEMPORAL ───────────────────────────────────────
df.select(
    F.min("tpep_pickup_datetime").alias("data_inicio"),
    F.max("tpep_pickup_datetime").alias("data_fim")
).show()

+-------------------+-------------------+
|        data_inicio|           data_fim|
+-------------------+-------------------+
|2002-12-31 23:06:55|2021-06-10 10:10:48|
+-------------------+-------------------+



In [56]:
invalid_dates = df.filter(
    (F.col("tpep_pickup_datetime") < "2020-01-01") |
    (F.col("tpep_pickup_datetime") > "2020-12-31")
).count()

print(f"Linhas com datas fora de 2020: {invalid_dates:,}")
print(f"Percentagem: {invalid_dates / total_rows * 100:.4f}%")

Linhas com datas fora de 2020: 45,072
Percentagem: 0.1829%


### **Outliers**

In [58]:
outlier_cols = [f.name for f in df.schema.fields
                if str(f.dataType) in ["DoubleType()", "LongType()", "IntegerType()"]]

quantiles = df.select([
    percentile_approx(c, 0.25).alias(f"{c}_q1") for c in outlier_cols
] + [
    percentile_approx(c, 0.75).alias(f"{c}_q3") for c in outlier_cols
]).collect()[0]

print(f"{'Column':<25} {'Q1':>10} {'Q3':>10} {'IQR':>10} {'Lower':>12} {'Upper':>12}")
print("-" * 80)

for c in outlier_cols:
    q1 = quantiles[f"{c}_q1"]
    q3 = quantiles[f"{c}_q3"]
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    print(f"{c:<25} {q1:>10.2f} {q3:>10.2f} {iqr:>10.2f} {lower:>12.2f} {upper:>12.2f}")

26/05/10 13:52:59 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Column                            Q1         Q3        IQR        Lower        Upper
--------------------------------------------------------------------------------
VendorID                        1.00       2.00       1.00        -0.50         3.50
passenger_count                 1.00       1.00       0.00         1.00         1.00
trip_distance                   0.99       3.00       2.01        -2.02         6.01
RatecodeID                      1.00       1.00       0.00         1.00         1.00
PULocationID                  114.00     234.00     120.00       -66.00       414.00
DOLocationID                  107.00     234.00     127.00       -83.50       424.50
payment_type                    1.00       2.00       1.00        -0.50         3.50
fare_amount                     6.50      14.00       7.50        -4.75        25.25
extra                           0.00       2.50       2.50        -3.75         6.25
mta_tax                         0.50       0.50       0.00         0.

Outlier detection was performed using the **IQR (Interquartile Range)** method, where values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR are considered outliers.

Several key findings emerge:

**`fare_amount`** has an upper bound of **$25.25**, meaning any trip costing more is statistically an outlier. However, airport trips and longer rides legitimately exceed this value, so a more permissive threshold of **$200** was applied instead of the strict IQR limit.

**`total_amount`** shows an upper bound of **$32.76**, with the same reasoning — legitimate trips can exceed this, particularly for airport runs or longer distances.

**`trip_distance`** has an upper bound of **6.01 miles**, which is consistent with typical NYC taxi trips. Values above this are statistically unusual but not necessarily erroneous — longer trips do occur. A hard cap of **1,000 miles** was applied to remove only clearly impossible values.

**`extra`**, **`tip_amount`** and **`tolls_amount`** all show lower bounds below zero, confirming the presence of negative values identified in the describe analysis.

**`mta_tax`**, `improvement_surcharge`, and `congestion_surcharge` show **zero IQR** — they are fixed-value fields with no variance, as expected.

**`passenger_count`** and `RatecodeID` also show zero IQR, indicating the vast majority of trips use standard rate (1) with 1 passenger.

### **Numerical Columns**

In [34]:
numeric_cols = [f.name for f in df.schema.fields
                if str(f.dataType) in ["DoubleType()", "LongType()", "IntegerType()"]]

df.select(numeric_cols).describe().show(truncate=False)

+-------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+---------------------+-----------------+--------------------+------------------+------------------+------------------+------------------+------------------+
|summary|VendorID          |passenger_count   |trip_distance     |RatecodeID        |PULocationID      |DOLocationID      |payment_type      |fare_amount      |extra             |mta_tax           |tip_amount        |tolls_amount      |improvement_surcharge|total_amount     |congestion_surcharge|trip_duration     |pickup_hour       |pickup_day_of_week|pickup_month      |fare_per_mile     |
+-------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+----------------

A statistical summary of all numerical attributes revealed several data quality issues that require attention before any analysis or modelling can be performed. Outlier detection was additionally performed using the **IQR (Interquartile Range)** method, where values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR are considered outliers.

#### Potentially Erroneous `fare_amount` and `total_amount`
Both fields present extreme outliers. `fare_amount` ranges from **-$1,259** to **$998,310**, and `total_amount` from **-$1,260** to **$1,000,003**. Negative values are physically impossible and likely result from system reversals or data entry errors. Values in the hundreds of thousands are equally implausible for NYC taxi trips. The IQR analysis confirms an upper bound of **$25.25** for `fare_amount` and **$32.76** for `total_amount`, reinforcing the presence of extreme outliers. These records will be removed by filtering `total_amount` to a reasonable range **(> $0 and < $10,000)**.

#### Unreasonable `trip_distance`
Trip distance ranges from **-30.62** to **350,914 miles** — the latter being equivalent to travelling around the Earth 14 times. Negative distances are impossible, and values above 1,000 miles are clearly erroneous. The IQR upper bound of **6.01 miles** confirms that the vast majority of NYC trips are short, making values above 1,000 miles unambiguously erroneous. Records with `trip_distance >= 1,000` will be removed.

#### Erroneous `passenger_count`
While the majority of trips have between 1 and 6 passengers, the field contains values of **0** (likely unrecorded by the driver) and up to **9**, which exceeds the legal NYC taxi capacity. The IQR shows zero variance — the median trip carries **1 passenger** — confirming that higher values are anomalous. Records with `passenger_count` of 0 or greater than 6 will be removed.

#### Implausible `trip_duration`
Trip duration ranges from **-531,231 minutes** to **8,525 minutes** (~6 days). Negative durations indicate dropoff timestamps recorded before pickup, which is impossible. Trips exceeding 3 hours (180 minutes) are also considered outliers for NYC taxi trips. Records outside the range **(1 to 180 minutes)** will be removed.

#### Anomalous `extra`, `mta_tax` and `fare_per_mile`
These fields also present negative minimum values and extreme maxima (e.g. `extra` reaching **$500,000**). The IQR confirms these as outliers — `extra` has an upper bound of **$6.25** under normal conditions. These anomalies are consistent with the same data entry or system errors identified above and will be addressed by the filters applied to `total_amount` and `trip_distance`.

#### Fixed-Value Fields
`mta_tax`, `improvement_surcharge`, and `congestion_surcharge` show **zero IQR**, confirming they are fixed-value fields with no variance, as expected from NYC TLC regulations.

# **4. Data Cleaning**

### **Invalid Dates**

In [15]:
df = df.filter(
    (F.col("tpep_pickup_datetime") >= "2020-01-01") &
    (F.col("tpep_pickup_datetime") <= "2020-12-31")
)

In [16]:
df.select(
    F.min("tpep_pickup_datetime").alias("data_inicio"),
    F.max("tpep_pickup_datetime").alias("data_fim")
).show()

+-------------------+-------------------+
|        data_inicio|           data_fim|
+-------------------+-------------------+
|2020-01-01 00:00:00|2020-12-31 00:00:00|
+-------------------+-------------------+



The dataset was filtered to retain only records within the year 2020. A total of **45,072 records** (0.18% of the dataset) presented `tpep_pickup_datetime` values outside this range — either predating 2020 or extending beyond December 31, 2020 — and were removed as 
invalid entries.

### **Problematic Records**
All problematic records are removed prior to analysis and modelling using a combined filter on `total_amount`, `trip_distance`, `passenger_count`, and `trip_duration`.

In [ ]:
before = df.count()

df = df.filter(
    (F.col("total_amount") > 0) &
    (F.col("total_amount") < 10000) &
    (F.col("trip_distance") >= 0) &
    (F.col("trip_distance") < 1000) &
    (F.col("passenger_count") > 0) &
    (F.col("passenger_count") <= 6) &
    (F.col("trip_duration") >= 1) &
    (F.col("trip_duration") <= 180)
)

after = df.count()

print(f"Before: {before:,}")
print(f"After:  {after:,}")
print(f"Removed: {before - after:,}")

### **Missing Values**

In [17]:
total_rows = df.count()

dfs = [
    df.select(
        F.lit(c).alias("column"),
        F.sum(F.col(c).isNull().cast("int")).alias("missing_count")
    )
    for c in df.columns
]

missing_table = reduce(lambda a, b: a.union(b), dfs) \
    .withColumn("missing_pct", F.round(F.col("missing_count") / total_rows * 100, 4)) \
    .orderBy("missing_count", ascending=False)

missing_table.show(truncate=False)

+---------------------+-------------+-----------+
|column               |missing_count|missing_pct|
+---------------------+-------------+-----------+
|passenger_count      |806563       |3.2782     |
|RatecodeID           |806563       |3.2782     |
|store_and_fwd_flag   |806563       |3.2782     |
|congestion_surcharge |806563       |3.2782     |
|VendorID             |0            |0.0        |
|tolls_amount         |0            |0.0        |
|total_amount         |0            |0.0        |
|tpep_pickup_datetime |0            |0.0        |
|tpep_dropoff_datetime|0            |0.0        |
|trip_distance        |0            |0.0        |
|PULocationID         |0            |0.0        |
|extra                |0            |0.0        |
|improvement_surcharge|0            |0.0        |
|DOLocationID         |0            |0.0        |
|payment_type         |0            |0.0        |
|fare_amount          |0            |0.0        |
|mta_tax              |0            |0.0        |


Profiling revealed missing values in **4 of the 18 attributes**. Four fields — `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, and `congestion_surcharge` — each present **806,563 null records** (~3.28% of the dataset), likely due to occasional failures in the taxi meter recording system.

The `airport_fee` field shows a near-total absence of values (**~99.99%**), which is expected: this surcharge was only introduced by the NYC TLC in 2022 and did not exist in 2020. This column will be dropped prior to analysis.

All remaining **14 attributes** contain no missing values.

In [18]:
# Preencher os nulos das restantes 4 colunas
df = df.fillna({
    "passenger_count": 0,
    "RatecodeID": 1,          # 1 = Standard rate (valor default NYC TLC)
    "store_and_fwd_flag": "N", # N = não foi armazenado
    "congestion_surcharge": 0
})

In [19]:
# ── 5. LINHAS COM PELO MENOS UM NULO ────────────────────────────
has_null = df.filter(
    F.greatest(*[F.col(c).isNull().cast("int") for c in df.columns]) == 1
).count()
print(f"Linhas com pelo menos 1 nulo: {has_null:,} ({has_null/total_rows*100:.1f}%)")

Linhas com pelo menos 1 nulo: 0 (0.0%)


### **Duplicate Records**

In [20]:
# ── 6. DUPLICADOS ───────────────────────────────────────────────
duplicates = total_rows - df.distinct().count()
print(f"Duplicados: {duplicates:,}")

Duplicados: 12,950


In [21]:
# Antes
total_before = df.count()

# Remover duplicados
df = df.distinct()

# Depois
total_after = df.count()

print(f"Linhas antes: {total_before:,}")
print(f"Linhas depois: {total_after:,}")
print(f"Duplicados removidos: {total_before - total_after:,}")

Linhas antes: 24,604,020
Linhas depois: 24,591,070
Duplicados removidos: 12,950


A total of **12,950 duplicate records** were identified and removed using `distinct()`. These exact-match duplicates are likely the result of data ingestion errors in the NYC TLC reporting system. After removal, the dataset retains **24,591,070 records**.

# **5. Feature Engineering**

In [22]:
# ── 7. FEATURE ENGINEERING ──────────────────────────────────────
df = df.withColumn("trip_duration",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60) \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime")) \
    .withColumn("pickup_month", F.month("tpep_pickup_datetime")) \
    .withColumn("is_rush_hour",
        ((F.hour("tpep_pickup_datetime").between(7, 9)) |
         (F.hour("tpep_pickup_datetime").between(16, 19))).cast("boolean")) \
    .withColumn("is_weekend",
        F.dayofweek("tpep_pickup_datetime").isin([1, 7]).cast("boolean")) \
    .withColumn("fare_per_mile",
        F.when(F.col("trip_distance") > 0, F.col("fare_amount") / F.col("trip_distance")))

print(f"Colunas após feature engineering: {len(df.columns)}")
df.printSchema()

Colunas após feature engineering: 25
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = false)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = false)
 |-- store_and_fwd_flag: string (nullable = false)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = false)
 |-- trip_duration: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_day_of_week: integer (nullable = true)
 

To support later querying and analysis tasks, **7 new features** were engineered from existing fields using `withColumn()` and Spark's built-in datetime functions.

- **`trip_duration`**: Duration of the trip in minutes, computed as the difference between `tpep_dropoff_datetime` and `tpep_pickup_datetime`.

- **`pickup_hour`**: Hour of the day extracted from `tpep_pickup_datetime`.

- **`pickup_day_of_week`**: Day of the week extracted from `tpep_pickup_datetime` (1 = Sunday, 7 = Saturday).

- **`pickup_month`**: Month extracted from `tpep_pickup_datetime`.

- **`is_rush_hour`**: Boolean indicator of whether the trip occurred during NYC peak traffic hours (7:00–9:00 and 16:00–19:00).

- **`is_weekend`**: Boolean indicator of whether the trip took place on a Saturday or Sunday.

- **`fare_per_mile`**: Fare amount divided by trip distance in miles, representing the cost efficiency of the trip. Only computed for trips with `trip_distance > 0`.

After feature engineering, the final dataset contains **25 attributes**.

## **Queries Examples**

In [37]:
import time

def run_query(query_name, query_sql):
    print(f"Running: {query_name}")
    start = time.time()

    result = spark.sql(query_sql)
    result.show()

    elapsed = time.time() - start
    print(f"⏱ {query_name} completed in {elapsed:.2f} seconds\n")
    return result

In [38]:
df.createOrReplaceTempView("taxi")

In [39]:
# Cache o DataFrame após a limpeza — evita reler os ficheiros em cada query
df.cache()
df.count()

22931642

In [40]:
# Query 1 - Average fare per payment type
run_query(
    "Average fare per payment type",
    """
    SELECT payment_type, ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM taxi
    GROUP BY payment_type
    ORDER BY avg_fare DESC
    """
)

# Query 2 - Top 10 busiest pickup zones by hour
run_query(
    "Top 10 busiest pickup zones by hour",
    """
    SELECT PULocationID, pickup_hour, COUNT(*) AS total_trips,
           ROUND(AVG(total_amount), 2) AS avg_total
    FROM taxi
    GROUP BY PULocationID, pickup_hour
    ORDER BY total_trips DESC
    LIMIT 10
    """
)

Running: Average fare per payment type


+------------+--------+
|payment_type|avg_fare|
+------------+--------+
|           5|   17.19|
|           4|   13.96|
|           3|   13.38|
|           1|   12.09|
|           2|    11.7|
+------------+--------+

⏱ Average fare per payment type completed in 1.20 seconds

Running: Top 10 busiest pickup zones by hour


+------------+-----------+-----------+---------+
|PULocationID|pickup_hour|total_trips|avg_total|
+------------+-----------+-----------+---------+
|         237|         14|      91068|    14.16|
|         237|         15|      89826|    14.33|
|         236|         15|      89122|    14.72|
|         237|         13|      82214|    13.96|
|         161|         18|      82114|    16.95|
|         161|         17|      81374|    17.41|
|         237|         17|      81164|    15.46|
|         236|         14|      80846|    14.41|
|         237|         16|      80554|    15.35|
|         237|         12|      80249|    13.73|
+------------+-----------+-----------+---------+

⏱ Top 10 busiest pickup zones by hour completed in 2.62 seconds



DataFrame[PULocationID: bigint, pickup_hour: int, total_trips: bigint, avg_total: double]

In [41]:
# Query 3 - Receita total e média por zona de pickup, hora e tipo de pagamento
run_query(
    "Revenue by pickup zone, hour and payment type",
    """
    SELECT
        PULocationID,
        pickup_hour,
        payment_type,
        COUNT(*) AS total_trips,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        ROUND(AVG(total_amount), 2) AS avg_revenue,
        ROUND(AVG(tip_amount), 2) AS avg_tip,
        ROUND(AVG(trip_distance), 2) AS avg_distance
    FROM taxi
    GROUP BY PULocationID, pickup_hour, payment_type
    HAVING COUNT(*) > 100
    ORDER BY total_revenue DESC
    LIMIT 20
    """
)

Running: Revenue by pickup zone, hour and payment type


+------------+-----------+------------+-----------+-------------+-----------+-------+------------+
|PULocationID|pickup_hour|payment_type|total_trips|total_revenue|avg_revenue|avg_tip|avg_distance|
+------------+-----------+------------+-----------+-------------+-----------+-------+------------+
|         132|         20|           1|      28164|   1771240.72|      62.89|   9.48|       17.06|
|         132|         16|           1|      25197|   1702536.25|      67.57|  10.16|       16.94|
|         132|         18|           1|      25678|   1677144.37|      65.31|   9.76|       16.84|
|         132|         21|           1|      25435|   1588647.73|      62.46|   9.38|       17.03|
|         132|         17|           1|      23701|   1582239.62|      66.76|  10.01|       17.01|
|         132|         15|           1|      23396|   1497025.06|      63.99|   9.56|       16.83|
|         132|         19|           1|      23121|   1491295.51|       64.5|   9.56|       16.98|
|         

DataFrame[PULocationID: bigint, pickup_hour: int, payment_type: bigint, total_trips: bigint, total_revenue: double, avg_revenue: double, avg_tip: double, avg_distance: double]

In [42]:
# Query 4 - Ranking das zonas por receita com window functions, comparando com a média mensal e identificando meses acima da média
run_query(
    "Zone revenue ranking vs monthly average with window functions",
    """
    WITH monthly_zone AS (
        SELECT
            PULocationID,
            pickup_month,
            COUNT(*) AS total_trips,
            ROUND(SUM(fare_amount), 2) AS total_fare,
            ROUND(AVG(fare_amount), 2) AS avg_fare,
            ROUND(AVG(trip_distance), 2) AS avg_distance
        FROM taxi
        GROUP BY PULocationID, pickup_month
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (PARTITION BY pickup_month ORDER BY total_fare DESC) AS rank_in_month,
            ROUND(AVG(total_fare) OVER (PARTITION BY pickup_month), 2) AS monthly_avg_fare,
            ROUND(AVG(total_fare) OVER (PARTITION BY PULocationID), 2) AS zone_avg_fare
        FROM monthly_zone
    )
    SELECT *,
        CASE WHEN total_fare > monthly_avg_fare THEN 'Above Average'
             ELSE 'Below Average' END AS performance
    FROM ranked
    WHERE rank_in_month <= 5
    ORDER BY pickup_month, rank_in_month
    """
)

Running: Zone revenue ranking vs monthly average with window functions


+------------+------------+-----------+----------+--------+------------+-------------+----------------+-------------+-------------+
|PULocationID|pickup_month|total_trips|total_fare|avg_fare|avg_distance|rank_in_month|monthly_avg_fare|zone_avg_fare|  performance|
+------------+------------+-----------+----------+--------+------------+-------------+----------------+-------------+-------------+
|         132|           1|     201760|8717131.11|   43.21|       14.93|            1|       295818.39|   2111644.91|Above Average|
|         138|           1|     127918|3751288.19|   29.33|        8.98|            2|       295818.39|    948052.02|Above Average|
|         161|           1|     273823|3088579.96|   11.28|         2.3|            3|       295818.39|    832544.18|Above Average|
|         230|           1|     220875|2651154.88|    12.0|        2.67|            4|       295818.39|    646402.61|Above Average|
|         237|           1|     284495|2574436.11|    9.05|        1.71|    

DataFrame[PULocationID: bigint, pickup_month: int, total_trips: bigint, total_fare: double, avg_fare: double, avg_distance: double, rank_in_month: int, monthly_avg_fare: double, zone_avg_fare: double, performance: string]

## **ML Model Example**

In [43]:
# ── Preparação e Cache ───────────────────────────────────────────
features = ["trip_distance", "pickup_hour", "passenger_count",
            "PULocationID", "DOLocationID", "RatecodeID"]

assembler = VectorAssembler(inputCols=features, outputCol="features")
df_ml = assembler.transform(df).select("features", "fare_amount")

df_ml.cache()
df_ml.count()

train, test = df_ml.randomSplit([0.8, 0.2], seed=42)
train.cache()
train.count()

evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction")

In [44]:
# ── Linear Regression ────────────────────────────────────────────
start_lr = time.time()
lr = LinearRegression(featuresCol="features", labelCol="fare_amount")
lr_model = lr.fit(train)
elapsed_lr = time.time() - start_lr

predictions_lr = lr_model.transform(test)
rmse_lr = evaluator.evaluate(predictions_lr, {evaluator.metricName: "rmse"})
r2_lr   = evaluator.evaluate(predictions_lr, {evaluator.metricName: "r2"})

print(f"Linear Regression → RMSE: {rmse_lr:.4f} | R²: {r2_lr:.4f} | Tempo: {elapsed_lr:.2f}s")


26/05/10 13:34:55 WARN Instrumentation: [e5655426] regParam is zero, which might cause numerical instability and overfitting.


Linear Regression → RMSE: 4.7375 | R²: 0.8051 | Tempo: 8.03s


In [45]:
# ── Random Forest ────────────────────────────────────────────────
start_rf = time.time()
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=50,
    maxDepth=10,
    seed=42
)
rf_model = rf.fit(train)
elapsed_rf = time.time() - start_rf

predictions_rf = rf_model.transform(test)
rmse_rf = evaluator.evaluate(predictions_rf, {evaluator.metricName: "rmse"})
r2_rf   = evaluator.evaluate(predictions_rf, {evaluator.metricName: "r2"})

print(f"Random Forest     → RMSE: {rmse_rf:.4f} | R²: {r2_rf:.4f} | Tempo: {elapsed_rf:.2f}s")


26/05/10 13:39:42 WARN DAGScheduler: Broadcasting large task binary with size 1023.1 KiB
26/05/10 13:40:22 WARN DAGScheduler: Broadcasting large task binary with size 1852.9 KiB
26/05/10 13:41:11 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/05/10 13:42:07 WARN DAGScheduler: Broadcasting large task binary with size 5.7 MiB
26/05/10 13:43:10 WARN DAGScheduler: Broadcasting large task binary with size 1562.8 KiB


Random Forest     → RMSE: 4.5108 | R²: 0.8233 | Tempo: 385.91s


In [46]:
# ── Comparação Final ─────────────────────────────────────────────
print(f"\n{'Modelo':<20} {'RMSE':>10} {'R²':>10} {'Tempo':>10}")
print("-" * 55)
print(f"{'Linear Regression':<20} {rmse_lr:>10.4f} {r2_lr:>10.4f} {elapsed_lr:>9.2f}s")
print(f"{'Random Forest':<20} {rmse_rf:>10.4f} {r2_rf:>10.4f} {elapsed_rf:>9.2f}s")


Modelo                     RMSE         R²      Tempo
-------------------------------------------------------
Linear Regression        4.7375     0.8051      8.03s
Random Forest            4.5108     0.8233    385.91s
